# 06 Quantization and the release gate

Dynamic 8-bit quantization stores the encoder's linear-layer weights as int8 and quantizes activations on the fly. It needs no training data and no GPU. The question is whether the classifier and the
gate survive it, so the answer is a paired comparison on the same test messages with an interval, and a tolerance written down **before** the run.

Then the **release gate**: a build fails unless the whole cascade meets targets set in advance for error, escalation rate, slow-tier calls, quantization loss and reproducibility.

In [1]:
import json, time, hashlib
import numpy as np
import torch
from pathlib import Path
from scipy.special import softmax
from scipy.stats import binomtest
from sklearn.linear_model import LogisticRegression
import meridian as m

RESULTS = Path("data/results")
r01 = json.loads((RESULTS / "01_encoders_and_baseline.json").read_text())
r03 = json.loads((RESULTS / "03_calibration_and_conformal.json").read_text())
r05 = json.loads((RESULTS / "05_langgraph_cascade.json").read_text())
ENC, T, ALPHA = r03["encoder"], r03["T"], r03["alpha_chosen"]; Q = r03[f"conformal_alpha_{ALPHA}"]["q"]
tr, te = m.load("train"), m.load("test")
_, ytr = m.texts_labels(tr); texts_te, yte = m.texts_labels(te); cl = m.cluster_ids(te)
E = m.cached_embeddings(ENC)
clf = LogisticRegression(C=r01[ENC]["C"], max_iter=3000).fit(E["train"], ytr)    # the head stays fp32 and fixed
torch.set_num_threads(4)
print("hardware:", m.hardware(), "| torch threads", torch.get_num_threads(), "| load average", [round(x, 1) for x in __import__('os').getloadavg()])

hardware: Intel(R) Core(TM) i7-7567U CPU @ 3.50GHz, 4 logical cores, Darwin 22.6.0 | torch threads 4 | load average [12.0, 35.4, 39.8]


## Targets, written before the measurement
These are the release criteria for the Meridian router. Changing them after seeing the result is how a gate stops being a gate.

In [2]:
TARGETS = {
    "cascade accuracy, Wilson lower bound >=": 0.93,
    "share of messages escalated to the LLM <=": 0.12,
    "share sent to a person <=": 0.05,
    "slow-tier calls per 1,000 messages <=": 120,
    "int8 minus fp32 accuracy, cluster-interval lower bound >=": -0.03,     # the quantization tolerance
    "int8 prediction agreement with fp32 >=": 0.97,
    "same input twice gives the same action and key (fraction) ==": 1.0,
}
for k, v in TARGETS.items(): print(f"  {k} {v}")

  cascade accuracy, Wilson lower bound >= 0.93
  share of messages escalated to the LLM <= 0.12
  share sent to a person <= 0.05
  slow-tier calls per 1,000 messages <= 120
  int8 minus fp32 accuracy, cluster-interval lower bound >= -0.03
  int8 prediction agreement with fp32 >= 0.97
  same input twice gives the same action and key (fraction) == 1.0


## Quantize the encoder

In [3]:
fp_tok, fp_mod = m.load_encoder(ENC)
q_mod = torch.quantization.quantize_dynamic(fp_mod, {torch.nn.Linear}, dtype=torch.qint8)
size = lambda mod: sum(p.numel() * p.element_size() for p in mod.parameters()) / 1e6
print(f"fp32 parameters: {size(fp_mod):.0f} MB | int8 model: linear layers quantized (parameter bytes shown for the remaining fp32 tensors: {size(q_mod):.0f} MB)")

def embed(mod, texts): return m.encode(ENC, texts, _loaded=(fp_tok, mod))
X_fp, X_q = embed(fp_mod, texts_te), embed(q_mod, texts_te)
cos = (X_fp * X_q).sum(1)
print(f"cosine between fp32 and int8 embeddings of the same message: median {np.median(cos):.4f}, min {cos.min():.4f}")

/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


fp32 parameters: 133 MB | int8 model: linear layers quantized (parameter bytes shown for the remaining fp32 tensors: 48 MB)


cosine between fp32 and int8 embeddings of the same message: median 0.9735, min 0.9387


### Accuracy before and after, paired

In [4]:
pf, pq = clf.predict(X_fp), clf.predict(X_q)
ok_f, ok_q = pf == yte, pq == yte
for name, ok in (("fp32", ok_f), ("int8", ok_q)):
    k, n = int(ok.sum()), len(ok); lo, hi = m.wilson(k, n); clo, chi = m.cluster_bootstrap_ci(ok.astype(float), cl)
    print(f"{name}: acc {k/n:.3f}  Wilson [{lo:.3f}, {hi:.3f}]  cluster [{clo:.3f}, {chi:.3f}]")
d = ok_q.astype(float) - ok_f.astype(float)
rng = np.random.default_rng(m.SEED); ids = np.unique(cl); per = {c: d[cl == c] for c in ids}
boots = [np.concatenate([per[c] for c in rng.choice(ids, len(ids))]).mean() for _ in range(5000)]
dlo, dhi = np.percentile(boots, [2.5, 97.5])
n10, n01 = int((ok_q & ~ok_f).sum()), int((~ok_q & ok_f).sum())
print(f"int8 minus fp32: {d.mean():+.3f}  cluster-bootstrap 95% [{dlo:+.3f}, {dhi:+.3f}] | int8-only right {n10}, fp32-only right {n01}, exact McNemar p = {binomtest(n10, n10+n01, .5).pvalue if n10+n01 else 1:.3f}")
agree = float((pf == pq).mean()); print(f"int8 and fp32 predict the same route on {agree:.1%} of messages")

# gate under quantization: same temperature, same conformal threshold
def gate_sizes(X): return (softmax(clf.decision_function(X) / T, 1) >= 1 - Q).sum(1)
sf, sq = gate_sizes(X_fp), gate_sizes(X_q)
print(f"gate decisions: fp32 act {np.mean(sf==1):.1%} / escalate {np.mean(sf>1):.1%} / person {np.mean(sf==0):.1%}  |  int8 act {np.mean(sq==1):.1%} / escalate {np.mean(sq>1):.1%} / person {np.mean(sq==0):.1%}")

fp32: acc 0.970  Wilson [0.936, 0.986]  cluster [0.930, 0.995]


int8: acc 0.950  Wilson [0.910, 0.973]  cluster [0.910, 0.985]


int8 minus fp32: -0.020  cluster-bootstrap 95% [-0.045, +0.000] | int8-only right 0, fp32-only right 4, exact McNemar p = 0.125
int8 and fp32 predict the same route on 98.0% of messages
gate decisions: fp32 act 91.5% / escalate 8.5% / person 0.0%  |  int8 act 89.0% / escalate 11.0% / person 0.0%


### Speed: batch of 32 and one message at a time

In [5]:
def throughput(mod, texts, reps=3):
    ts = []
    for _ in range(reps):
        t = time.perf_counter(); m.encode(ENC, texts, batch_size=32, _loaded=(fp_tok, mod)); ts.append(time.perf_counter() - t)
    return len(texts) / np.median(ts)
tp_f, tp_q = throughput(fp_mod, texts_te), throughput(q_mod, texts_te)
print(f"batch 32 throughput: fp32 {tp_f:.0f} msg/s | int8 {tp_q:.0f} msg/s | ratio {tp_q/tp_f:.2f}")
one = texts_te[:100]
lf = m.time_calls(lambda t: m.encode(ENC, [t], _loaded=(fp_tok, fp_mod)), one)
lq = m.time_calls(lambda t: m.encode(ENC, [t], _loaded=(fp_tok, q_mod)), one)
print(f"batch 1 latency: fp32 median {lf['median_ms']:.1f} ms, p95 {lf['p95_ms']:.1f} ms | int8 median {lq['median_ms']:.1f} ms, p95 {lq['p95_ms']:.1f} ms")
print("uptime load (1 / 5 / 15 min):", [round(x, 1) for x in __import__('os').getloadavg()], "<- speeds are only comparable within this run")

batch 32 throughput: fp32 43 msg/s | int8 59 msg/s | ratio 1.35


batch 1 latency: fp32 median 29.5 ms, p95 38.9 ms | int8 median 32.0 ms, p95 65.9 ms
uptime load (1 / 5 / 15 min): [9.0, 30.9, 37.9] <- speeds are only comparable within this run


## Reproducibility: the same input, twice
For every test message, compute the action and its idempotency key twice from scratch. They must be identical.

In [6]:
def action_and_key(text, mid, mod):
    p = softmax(clf.decision_function(m.encode(ENC, [text], _loaded=(fp_tok, mod))) / T, 1)[0]
    s = [m.ROUTES[i] for i in np.where(p >= 1 - Q)[0]]
    action = f"queue:{s[0]}" if len(s) == 1 else "escalate" if len(s) > 1 else "person"
    key = hashlib.sha256(f"{mid}|gate-v1:alpha={ALPHA}|{action}".encode()).hexdigest()[:16]
    return action, key
same = np.mean([action_and_key(r["text"], r["id"], fp_mod) == action_and_key(r["text"], r["id"], fp_mod) for r in te])
print(f"identical action and key on {same:.1%} of {len(te)} messages (same machine, same threads; not a claim across machines)")

identical action and key on 100.0% of 200 messages (same machine, same threads; not a claim across machines)


## The release gate
The candidate is a whole build, not a number. For each candidate (fp32 encoder, int8 encoder) we replay the cascade on the test split with that encoder: one route in the set acts,
several routes use the recorded slow-tier reply, none goes to a person. Then every check compares a measured number with the target written above.

In [7]:
idmap = json.loads(Path("data/llm/test_id_map.json").read_text())
llm = {idmap[j["id"]]: j["route"] for j in map(json.loads, Path("data/llm/test_haiku_answers.jsonl").read_text().splitlines())}

def cascade_eval(X):
    P = softmax(clf.decision_function(X) / T, 1); S = P >= 1 - Q; size = S.sum(1)
    final, by = [], []
    for i, r in enumerate(te):
        if size[i] == 1: final.append(int(np.where(S[i])[0][0])); by.append("encoder")
        elif size[i] > 1 and llm.get(r["id"]) in m.ROUTES: final.append(m.ROUTE_ID[llm[r["id"]]]); by.append("llm")
        else: final.append(None); by.append("person")
    ans = np.array([f is not None for f in final]); ok = np.array([f == y for f, y in zip(final, yte)])[ans]
    k, n = int(ok.sum()), int(ans.sum())
    return dict(acc=k / n, wilson_lo=m.wilson(k, n)[0], escalated=float(np.mean(np.array(by) == "llm")), person=float(np.mean(np.array(by) == "person")),
                calls_per_1000=float(np.mean(np.array(by) == "llm") * 1000))
cand = {"fp32 build": cascade_eval(X_fp), "int8 build": cascade_eval(X_q)}
for name, c in cand.items(): print(f"{name}: cascade accuracy {c['acc']:.3f} (Wilson lower {c['wilson_lo']:.3f}) | escalated {c['escalated']:.1%} | person {c['person']:.1%} | slow-tier calls per 1,000: {c['calls_per_1000']:.0f}")

def gate(name, c, quantized):
    rows = [
        ("cascade accuracy, Wilson lower bound >=", c["wilson_lo"], lambda v: v >= TARGETS["cascade accuracy, Wilson lower bound >="]),
        ("share of messages escalated to the LLM <=", c["escalated"], lambda v: v <= TARGETS["share of messages escalated to the LLM <="]),
        ("share sent to a person <=", c["person"], lambda v: v <= TARGETS["share sent to a person <="]),
        ("slow-tier calls per 1,000 messages <=", c["calls_per_1000"], lambda v: v <= TARGETS["slow-tier calls per 1,000 messages <="]),
        ("same input twice gives the same action and key (fraction) ==", float(same), lambda v: v == 1.0),
    ]
    if quantized:
        rows += [("int8 minus fp32 accuracy, cluster-interval lower bound >=", dlo, lambda v: v >= TARGETS["int8 minus fp32 accuracy, cluster-interval lower bound >="]),
                 ("int8 prediction agreement with fp32 >=", agree, lambda v: v >= TARGETS["int8 prediction agreement with fp32 >="])]
    print(f"\n--- gate for the {name} ---")
    allok = True
    for label, v, f in rows:
        ok = bool(f(v)); allok &= ok; print(f"{'PASS' if ok else 'FAIL'}  {label}   measured {v:.4f}")
    print(f"RELEASE GATE, {name}:", "PASS" if allok else "FAIL")
    return allok, [(l, float(v), bool(f(v))) for l, v, f in rows]
gate_fp, rows_fp = gate("fp32 build", cand["fp32 build"], False)
gate_q, rows_q = gate("int8 build", cand["int8 build"], True)
passed = gate_fp and gate_q

res = dict(fp32_acc=float(ok_f.mean()), int8_acc=float(ok_q.mean()), diff=float(d.mean()), diff_cluster_ci=[float(dlo), float(dhi)], agreement=agree,
           mcnemar=[n10, n01], throughput_fp32=tp_f, throughput_int8=tp_q, latency_fp32=lf, latency_int8=lq,
           gate_fp32=[float(np.mean(sf == 1)), float(np.mean(sf > 1)), float(np.mean(sf == 0))], gate_int8=[float(np.mean(sq == 1)), float(np.mean(sq > 1)), float(np.mean(sq == 0))],
           reproducible_fraction=float(same), candidates=cand, gate_fp32_build=dict(passed=gate_fp, checks=rows_fp), gate_int8_build=dict(passed=gate_q, checks=rows_q),
           hardware=m.hardware(), load=list(__import__('os').getloadavg()), torch_threads=torch.get_num_threads())
(RESULTS / "06_quantization_and_release.json").write_text(json.dumps(res, indent=2, default=float))

fp32 build: cascade accuracy 0.975 (Wilson lower 0.943) | escalated 8.5% | person 0.0% | slow-tier calls per 1,000: 85
int8 build: cascade accuracy 0.975 (Wilson lower 0.943) | escalated 11.0% | person 0.0% | slow-tier calls per 1,000: 110

--- gate for the fp32 build ---
PASS  cascade accuracy, Wilson lower bound >=   measured 0.9428
PASS  share of messages escalated to the LLM <=   measured 0.0850
PASS  share sent to a person <=   measured 0.0000
PASS  slow-tier calls per 1,000 messages <=   measured 85.0000
PASS  same input twice gives the same action and key (fraction) ==   measured 1.0000
RELEASE GATE, fp32 build: PASS

--- gate for the int8 build ---
PASS  cascade accuracy, Wilson lower bound >=   measured 0.9428
PASS  share of messages escalated to the LLM <=   measured 0.1100
PASS  share sent to a person <=   measured 0.0000
PASS  slow-tier calls per 1,000 messages <=   measured 110.0000
PASS  same input twice gives the same action and key (fraction) ==   measured 1.0000
FAIL  

2466